# 08 各群平均趨勢折線與增長摘要

**用途**：依分群結果畫各群平均發生率折線，並彙整 1979–2022 的增長摘要。

| 項目 | 說明 |
|---|---|
| 輸入 | 分群結果（於本檔重算） |
| 輸出 | 各群趨勢圖與增長摘要（文字） |
| 來源 | MVA 期末報告分析.ipynb（Colab 原檔，191 cells） 之 cell 174–183（Update 05/29 最終版） |


In [ ]:
# 共用設定：載入資料並提供 df / pivot_table_age_std / pivot_table_cleaned / scaled_data / taiwan_map / prop 等變數
# （取代原 Colab 的 Drive 掛載、字型下載與 GDAL 設定；詳見 _legacy_setup.py）
import sys; sys.path.insert(0, '..')   # _legacy_setup.py 位於 notebooks/
from _legacy_setup import *

In [ ]:
# 前置狀態（原 Colab 於 05–06 產生）
import shutil
if not (SCRATCH / 'hierarchical_clustering_comprehensive_results.csv').exists():
    shutil.copy(ROOT / 'outputs/tables/hclust_cluster_assignments.csv', SCRATCH / 'hierarchical_clustering_comprehensive_results.csv')
_k4 = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(scaled_data)
df_scores = pd.DataFrame({'County': pivot_table_cleaned.index, 'Cluster': [f'Group {chr(65 + l)}' for l in _k4]})

### Lineplots by Clusters

In [ ]:
# 1. 確保已剔除連江縣，並建立包含分群標籤的 DataFrame
df_plot_raw = pivot_table_age_std.drop(index='連江縣', errors='ignore').copy()

# 假設 df_scores 是我們上一步做好的 DataFrame (內含 County 和 Cluster 欄位)
# 將分群資訊 mapping 回原始的 pivot table
df_plot_raw = df_plot_raw.reset_index()
df_plot_raw = df_plot_raw.merge(df_scores[['County', 'Cluster']], left_on='City_County', right_on='County')

# 2. 將資料由寬轉長 (Melt)，方便 Seaborn 畫折線圖
df_long = df_plot_raw.drop(columns=['County']).melt(
    id_vars=['City_County', 'Cluster'],
    var_name='Year',
    value_name='Incidence_Rate'
)
df_long['Year'] = df_long['Year'].astype(int)

#### K-Means

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

# 排除連江縣 (確保與地圖一致)
df_long_clean = df_long[df_long['City_County'] != '連江縣'].copy()

# ==========================================
# 1. 動態計算 K-Means 標籤 (k=2 與 k=5)
# ==========================================
# K=2 分群
km_2 = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_k2 = km_2.fit_predict(scaled_data)
map_k2 = {i: f'Group {chr(65 + i)}' for i in range(2)}
df_label_k2 = pd.DataFrame({'County': pivot_table_cleaned.index, 'Cluster_k2': [map_k2[l] for l in labels_k2]})

# K=5 分群 (✨ 已依需求由 k=4 改為 k=5)
km_5 = KMeans(n_clusters=5, random_state=42, n_init=10)
labels_k5 = km_5.fit_predict(scaled_data)
map_k5 = {i: f'Group {chr(65 + i)}' for i in range(5)}
df_label_k5 = pd.DataFrame({'County': pivot_table_cleaned.index, 'Cluster_k5': [map_k5[l] for l in labels_k5]})

# 將分群標籤合併回長資料 (Long format)
df_plot_data = df_long_clean.merge(df_label_k2, left_on='City_County', right_on='County', how='left')
df_plot_data = df_plot_data.merge(df_label_k5, left_on='City_County', right_on='County', how='left')

# ==========================================
# 2. 定義統一的主色彩字典 (擴充至 Group E)
# ==========================================
color_dict = {
    'Group A': '#e41a1c',  # 紅
    'Group B': '#377eb8',  # 藍
    'Group C': '#4daf4a',  # 綠
    'Group D': '#984ea3',  # 紫
    'Group E': '#ff7f00'   # 橘 (✨ k=5 新增)
}

# 中文字型安全設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# ==========================================
# 3. 繪製圖表一：K-Means (k=2) - 獨立畫布
# ==========================================
k_val = 2
df_means_k2 = df_plot_data.groupby(['Year', 'Cluster_k2'])['Incidence_Rate'].mean().reset_index()

fig1, ax1 = plt.subplots(figsize=(14, 8))

# A. 背景：各縣市極淡細線
sns.lineplot(data=df_plot_data, x='Year', y='Incidence_Rate', hue='Cluster_k2', units='City_County',
             estimator=None, palette=color_dict, linewidth=0.8, alpha=0.15, legend=False, ax=ax1)

# B. 前景：群組平均粗線 (✨ 已採用你的小小更動設定)
sns.lineplot(data=df_means_k2, x='Year', y='Incidence_Rate', hue='Cluster_k2',
             palette=color_dict, linewidth=2.5, marker='o', markersize=5.5, ax=ax1)

# C. 標準化介面與圖例
ax1.set_title(f'各分群年度平均肺癌發生率：K-Means (k={k_val})', fontsize=14, fontweight='bold', pad=15)
ax1.set_ylabel('平均發生率 (每10萬人口)', fontsize=11)
ax1.set_xlabel('診斷年份 (Year)', fontsize=11)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(title=f'Cluster (k={k_val})', loc='upper left', framealpha=0.8)

plt.tight_layout()
plt.savefig(f'trends_kmeans_k{k_val}.png', dpi=300, bbox_inches='tight')
plt.show()


# ==========================================
# 4. 繪製圖表二：K-Means (k=5) - 獨立畫布 (✨ 全新獨立繪製)
# ==========================================
k_val = 5
df_means_k5 = df_plot_data.groupby(['Year', 'Cluster_k5'])['Incidence_Rate'].mean().reset_index()

fig2, ax2 = plt.subplots(figsize=(14, 8))

# A. 背景：各縣市極淡細線
sns.lineplot(data=df_plot_data, x='Year', y='Incidence_Rate', hue='Cluster_k5', units='City_County',
             estimator=None, palette=color_dict, linewidth=0.8, alpha=0.15, legend=False, ax=ax2)

# B. 前景：群組平均粗線 (✨ 同步套用你的更動設定)
sns.lineplot(data=df_means_k5, x='Year', y='Incidence_Rate', hue='Cluster_k5',
             palette=color_dict, linewidth=2.5, marker='o', markersize=5.5, ax=ax2)

# C. 標準化介面與圖例
ax2.set_title(f'各分群年度平均肺癌發生率：K-Means (k={k_val})', fontsize=14, fontweight='bold', pad=15)
ax2.set_ylabel('平均發生率 (每10萬人口)', fontsize=11)
ax2.set_xlabel('診斷年份 (Year)', fontsize=11)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(title=f'Cluster (k={k_val})', loc='upper left', framealpha=0.8)

plt.tight_layout()
plt.savefig(f'trends_kmeans_k{k_val}.png', dpi=300, bbox_inches='tight')
plt.show()


# ==========================================
# 5. 數據摘要輸出 (✨ 擴充同時顯示 k=2 與 k=5 的結果)
# ==========================================
print("\n" + "="*50)
print("   📊 肺癌平均發生率增長摘要 (1979-2022)   ")
print("="*50)

# 輸出 K=2 摘要
print("\n🔹 【K-Means 分群結果 (k=2)】")
for group in sorted(df_means_k2['Cluster_k2'].unique()):
    subset = df_means_k2[df_means_k2['Cluster_k2'] == group]
    start, end = subset.iloc[0]['Incidence_Rate'], subset.iloc[-1]['Incidence_Rate']
    pct = ((end - start) / start) * 100
    print(f"  - {group}: {start:.2f} -> {end:.2f} ({pct:+.1f}%)")

# 輸出 K=5 摘要
print("\n🔸 【K-Means 分群結果 (k=5)】")
for group in sorted(df_means_k5['Cluster_k5'].unique()):
    subset = df_means_k5[df_means_k5['Cluster_k5'] == group]
    start, end = subset.iloc[0]['Incidence_Rate'], subset.iloc[-1]['Incidence_Rate']
    pct = ((end - start) / start) * 100
    print(f"  - {group}: {start:.2f} -> {end:.2f} ({pct:+.1f}%)")
print("\n" + "="*50)

K-Means 分群的增長摘要


```
==================================================
   📊 肺癌平均發生率增長摘要 (1979-2022)   
==================================================
🔹 【K-Means 分群結果 (k=2)】
  - Group A: 11.05 -> 42.22 (+282.1%)
  - Group B: 5.00 -> 32.23 (+544.2%)

🔸 【K-Means 分群結果 (k=5)】
  - Group A: 9.77 -> 41.90 (+328.8%)
  - Group B: 2.40 -> 16.92 (+605.0%) (金門)
  - Group C: 5.66 -> 36.06 (+537.7%)
  - Group D: 19.42 -> 41.86 (+115.5%)
  - Group E: 1.27 -> 47.21 (+3617.3%) (澎湖)
==================================================
```



#### H-Clustering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 資料清理與安全對齊 (Data Preparing)
# ==========================================
# 排除連江縣 (確保與地圖、K-Means 的基底完全一致)
df_long_clean = df_long[df_long['City_County'] != '連江縣'].copy()

# 讀取先前儲存的 16 種組合層次聚類結果 (以縣市名稱為 Index)
df_hierarchical = pd.read_csv('hierarchical_clustering_comprehensive_results.csv', index_col=0)

# ==========================================
# 2. 定義統一的主色彩字典 (與地圖、K-Means 保持絕對一致)
# ==========================================
color_dict = {
    'Group A': '#e41a1c',  # 紅
    'Group B': '#377eb8',  # 藍
    'Group C': '#4daf4a',  # 綠
    'Group D': '#984ea3',  # 紫
    'Group E': '#ff7f00'   # 橘
}

# 中文字型安全設定 (沿用您先前的設定)
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# ==========================================
# 3. 配置方案 A 與方案 B 的繪圖任務清單 (Task Configuration)
# ==========================================
map_tasks = [
    # 方案 A：所有方法皆看 k = 2
    {'scenario': 'A', 'method': 'Single',   'k': 2},
    {'scenario': 'A', 'method': 'Complete', 'k': 2},
    {'scenario': 'A', 'method': 'Average',  'k': 2},
    {'scenario': 'A', 'method': 'Ward',     'k': 2},

    # 方案 B：指定最佳/次佳群數 (前三者 k=4, Ward k=5)
    {'scenario': 'B', 'method': 'Single',   'k': 4},
    {'scenario': 'B', 'method': 'Complete', 'k': 4},
    {'scenario': 'B', 'method': 'Average',  'k': 4},
    {'scenario': 'B', 'method': 'Ward',     'k': 5},
]

# 建立一個字典，用來動態收集各任務的摘要數據，最後統一漂亮輸出
summary_logs = {}

# ==========================================
# 4. 進入自動化繪圖迴圈 (共 8 張圖獨立畫布)
# ==========================================
for task in map_tasks:
    scenario = task['scenario']
    method = task['method']
    k_val = task['k']

    # 對齊 CSV 的分群標籤欄位名稱（例如：'Single_k2', 'Ward_k5'）
    csv_col_name = f"{method}_k{k_val}"

    # 建立暫時的對照 DataFrame
    df_temp = pd.DataFrame({
        'County': df_hierarchical.index,
        'Cluster_Label': df_hierarchical[csv_col_name]
    })

    # 將該演算法的分群標籤動態合併進長資料中
    df_task_data = df_long_clean.merge(df_temp, left_on='City_County', right_on='County', how='left')

    # 即時計算該組合在各年度的「群組平均發生率 (Cluster Means)」
    df_task_means = df_task_data.groupby(['Year', 'Cluster_Label'])['Incidence_Rate'].mean().reset_index()

    # 儲存摘要數據供後續輸出
    task_key = f"Scenario {scenario}: {method} Linkage (k={k_val})"
    summary_logs[task_key] = df_task_means

    # ------------------------------------------
    # 建立獨立畫布 (每一張圖都是全新的獨立 Figure)
    # ------------------------------------------
    fig, ax = plt.subplots(figsize=(14, 8))

    # 層次一：背景（所有個別縣市的「極淡細線」）
    # legend=False 確保個別縣市名稱不會污染圖例
    sns.lineplot(data=df_task_data, x='Year', y='Incidence_Rate', hue='Cluster_Label', units='City_County',
                 estimator=None, palette=color_dict, linewidth=0.8, alpha=0.15, legend=False, ax=ax)

    # 層次二：前景（群組平均的「飽和粗線」+ 顯眼轉折圓點）
    # 這裡採用了您的黃金比例參數修正：linewidth=2.5, markersize=5.5
    sns.lineplot(data=df_task_means, x='Year', y='Incidence_Rate', hue='Cluster_Label',
                 palette=color_dict, linewidth=2.5, marker='o', markersize=5.5, ax=ax)

    # ------------------------------------------
    # 5. 介面與圖例標準化設定
    # ------------------------------------------
    # 標題標準化：各分群年度平均肺癌發生率：Hierarchical - {Method} (k={k})
    ax.set_title(f'各分群年度平均肺癌發生率：Hierarchical - {method} (k={k_val})',
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_ylabel('平均發生率 (每10萬人口)', fontsize=11)
    ax.set_xlabel('診斷年份 (Year)', fontsize=11)
    ax.grid(True, linestyle=':', alpha=0.6)

    # 圖例標準化：標題設為 Cluster (k=k_val)
    ax.legend(title=f'Cluster (k={k_val})', loc='upper left', framealpha=0.8)

    # 檔案命名標準化並安全儲存
    file_name = f'trend_hierarchical_{method}_k{k_val}.png'
    plt.tight_layout()
    plt.savefig(file_name, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"📸 獨立趨勢圖已成功儲存：{file_name}")

# ==========================================
# 6. 數據摘要動態輸出 (Growth Summary Logging)
# ==========================================
print("\n" + "="*60)
print("   📊 層次聚類分群：肺癌平均發生率增長摘要 (1979-2022)   ")
print("="*60)

for task_title, df_means in summary_logs.items():
    print(f"\n🔹 【{task_title}】")
    for group in sorted(df_means['Cluster_Label'].unique()):
        subset = df_means[df_means['Cluster_Label'] == group]

        # 安全機制：防範某些極端演算法分群中出現空資料
        if not subset.empty:
            start, end = subset.iloc[0]['Incidence_Rate'], subset.iloc[-1]['Incidence_Rate']
            pct = ((end - start) / start) * 100
            print(f"  - {group}: {start:.2f} -> {end:.2f} ({pct:+.1f}%)")
        else:
            print(f"  - {group}: 無資料")

print("\n" + "="*60)
print("🎉 所有層次聚類學術折線圖（共 8 張）與增長統計摘要已全部生成完畢！")

Hierarchical Clustering 的增長摘要

```
============================================================
   📊 階層式分群：肺癌平均發生率增長摘要 (1979-2022)   
============================================================
🔹 【Scenario A: Single Linkage (k=2)】
  - Group A: 9.97 -> 40.99 (+311.1%)
  - Group B: 2.40 -> 16.92 (+605.0%)

🔹 【Scenario A: Complete Linkage (k=2)】
  - Group A: 9.97 -> 40.99 (+311.1%)
  - Group B: 2.40 -> 16.92 (+605.0%)

🔹 【Scenario A: Average Linkage (k=2)】
  - Group A: 9.97 -> 40.99 (+311.1%)
  - Group B: 2.40 -> 16.92 (+605.0%)

🔹 【Scenario A: Ward Linkage (k=2)】
  - Group A: 11.05 -> 42.22 (+282.1%)
  - Group B: 5.00 -> 32.23 (+544.2%)
------------------------------------------------------------
🔹 【Scenario B: Single Linkage (k=4)】
  - Group A: 11.70 -> 41.89 (+258.0%)
  - Group B: 5.66 -> 36.06 (+537.7%)
  - Group C: 1.27 -> 47.21 (+3617.3%)
  - Group D: 2.40 -> 16.92 (+605.0%)

🔹 【Scenario B: Complete Linkage (k=4)】
  - Group A: 12.08 -> 42.58 (+252.4%)
  - Group B: 2.40 -> 16.92 (+605.0%)
  - Group C: 6.84 -> 36.51 (+433.5%)
  - Group D: 1.27 -> 47.21 (+3617.3%)

🔹 【Scenario B: Average Linkage (k=4)】
  - Group A: 11.70 -> 41.89 (+258.0%)
  - Group B: 5.66 -> 36.06 (+537.7%)
  - Group C: 1.27 -> 47.21 (+3617.3%)
  - Group D: 2.40 -> 16.92 (+605.0%)

🔹 【Scenario B: Ward Linkage (k=5)】
  - Group A: 9.77 -> 41.90 (+328.8%)
  - Group B: 5.66 -> 36.06 (+537.7%)
  - Group C: 19.42 -> 41.86 (+115.5%)
  - Group D: 2.40 -> 16.92 (+605.0%)
  - Group E: 1.27 -> 47.21 (+3617.3%)
============================================================
```



### 小節

一、

如果您仔細對比以下兩組數據：

* 【 K-Means (k=2) 】 與 【 Hierarchical: Ward (k=2) 】
* 【 K-Means (k=5) 】 與 【 Hierarchical: Ward (k=5) 】

您會發現它們的起始值、結束值與增長率完全一模一樣！

**解讀**：這在統計學上是一個非常漂亮的實證。因為 Ward's Method（最小變異法） 的核心優化目標就是「最小化群內的平方誤差和（SSE）」，這與 K-Means 的目標函數在數學本質上是完全一致的。這項結果有力地證明了這兩個截然不同的演算法（一個是迭代分割、一個是凝聚層次），在空間分布與趨勢切分上達到了完美的收斂。

二、

看 Scenario A (k=2) 時，Single、Complete、Average 三種聯結法產出的群組數據完全一致：

Group A: 9.97 $\rightarrow$ 40.99

Group B: 2.40 $\rightarrow$ 16.92（這條線其實就是「金門縣」的獨立軌跡）

**解讀**：這揭示了傳統層次聚類在 $k$ 值過小時的經典缺陷——連鎖效應（Chaining Effect）或對極端值的過度敏感。除了金門縣（Group B）因為早期發生率極低且成長平緩被孤立出來之外，全台灣其他 20 個縣市不論本島、外島、南北差異，全部被硬生生塞進了 Group A。這證明了在 $k=2$ 時，這三種方法無法提供有意義的地理宏觀分群，唯獨 Ward 法 成功將台灣切分成均勻的兩大宏觀地理板塊（Group A 與 B）。

---

📝 報告結論寫作建議（Takeaway）

在您的報告中，強烈建議推崇 K-Means (k=5) 與 Hierarchical: Ward Linkage (k=5) 作為本次研究的最佳視覺化與分群模型。

寫作範例：
本研究對比了多種聚類演算法在 1979-2022 年台灣肺癌發生率長軌跡數據上的表現。實驗結果顯示，Single、Complete、Average 聯結法在小群數時極易受到金門、澎湖等外島極端歷史軌跡的干擾，產生嚴重的孤立群組（Chaining Effect），無法有效剖析台灣本島的地理流行病學結構。

相反地，Ward 最小變異法（k=5）與 K-Means（k=5）呈現了完全一致的收斂結果。該模型不僅精準捕捉了增長高達 +3617.3% 的極端突變型縣市（Group E），更成功將台灣本島依據「歷史高基數飽和型（Group C, +115.5%）」、「持續高攀型（Group A）」與「後期爆發型（Group B, +537.7%）」進行了具有高度公衛防癌實政價值的多層次切分，最適合做為後續資源配置的決策依據。